# Feature engineering
This script adds new features to the dataset for the Econ-ML project.

#### Libraries

In [ ]:
import pandas as pd


Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
import_sup_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Initial/Gravity_dta_V202211/'
export_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'

Data uploading

In [ ]:
merged_main = pd.read_csv(import_path + "merged_main_raw.csv",
                          dtype={"exp_iso3": str, "imp_iso3": str,
                                 "descr_trade": str, "objective": str})
merged_robust = pd.read_csv(import_path + "merged_robust_raw.csv",
                            dtype={"exp_iso3": str, "imp_iso3": str,
                                   "descr_trade": str, "objective": str})

/tmp/ipykernel_4589/1129118814.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_main = pd.read_csv(import_path + "merged_main_raw.csv",
/tmp/ipykernel_4589/1129118814.py:4: DtypeWarning: Columns (23,29) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_robust = pd.read_csv(import_path + "merged_robust_raw.csv",


### Data pry and preparation

Quick terminology note:

Sender = the country that imposes the sanction (sanctioning_state_iso3).


Target = the country the sanction is imposed on (sanctioned_state_iso3).

In [ ]:
# type of export/import limitations
print(merged_main["descr_trade"].value_counts(dropna=False))

descr_trade
NaN                    1558964
exp_part,imp_part         9949
exp_part                  7325
imp_part                  6514
exp_compl,imp_compl       4970
exp_part,imp_compl          27
imp_compl                    9
exp_compl                    2
exp_compl,imp_part           2
Name: count, dtype: int64


descr_exp_compl = 1 if the sanction completely bans the sender's exports to the target (a full export embargo).


descr_imp_compl = 1 if it completely bans the sender's imports from the target (a full import ban).


descr_exp_part = 1 if it partially restricts the sender's exports to the target (some goods/sectors, not all).


descr_imp_part = 1 if it partially restricts the sender's imports from the target.

In [ ]:
# sanctions objective (aim - to force target to do smth)
print(merged_main["objective"].value_counts(dropna=False).head(30))

objective
NaN                                                               1489227
end_war                                                             10238
end_war,human_rights                                                 8210
prevent_war                                                          7605
end_war,end_war                                                      7024
human_rights                                                         4594
democracy                                                            4506
terrorism,end_war                                                    3115
democracy,human_rights                                               2808
end_war,human_rights,end_war,human_rights                            2700
policy_change                                                        2463
end_war,prevent_war                                                  2422
end_war,end_war,end_war                                              1957
other                       

In [ ]:
# Regional Trade Agreement between countries (and in what categories)
print(merged_main["rta"].value_counts(dropna=False).head(30))

rta
NaN                 1457643
Goods                 74631
Goods & Services      55266
Services                222
Name: count, dtype: int64


Not as tidy as the export/import restrictions, will clean soon.

In [ ]:
# see the token for export/import rest. and objective
def tokens(series):
    out = set()
    for v in series.dropna():
        out |= {t.strip() for t in str(v).split(",")}
    return sorted(out)

print("descr_trade tokens:", tokens(merged_main["descr_trade"]))
print("objective tokens:  ", tokens(merged_main["objective"]))

descr_trade tokens: ['exp_compl', 'exp_part', 'imp_compl', 'imp_part']
objective tokens:   ['democracy', 'destab_regime', 'end_war', 'human_rights', 'other', 'policy_change', 'prevent_war', 'territorial_conflict', 'terrorism']


### Feature creation

In [ ]:
# ============================================================
# Sanction feature encoding
#   - descr_trade : multi-label -> 4 direction/intensity dummies
#   - objective   : multi-label, dedup repeats -> one dummy per objective
#   - rta         : categorical control -> one-hot (rta - )
# Applied identically to merged_main and merged_robust.
# ============================================================

def token_dummies(df, col, prefix):
    """Split a comma-joined multi-label column into 0/1 dummies,
    one per UNIQUE token (repeats within a cell collapse to one)."""
    s = df[col].fillna("").astype(str)
    # unique tokens per row (dedupes 'end_war,end_war' -> {'end_war'})
    token_sets = s.apply(lambda v: {t.strip() for t in v.split(",") if t.strip()})
    vocab = sorted(set().union(*token_sets)) if len(token_sets) else []
    for tok in vocab:
        df[f"{prefix}_{tok}"] = token_sets.apply(lambda ts: int(tok in ts))
    return df, vocab

def add_sanction_features(df):
    df = df.copy()

    # 1) descr_trade -> descr_exp_compl / descr_imp_compl / descr_exp_part / descr_imp_part
    df, trade_vocab = token_dummies(df, "descr_trade", "descr")

    # 2) objective -> one dummy per distinct objective (repeats deduped)
    df, obj_vocab = token_dummies(df, "objective", "obj")

    # 3) rta categorical control -> dummies (NaN = 'no RTA' = all zeros = base)
    rta_dummies = pd.get_dummies(df["rta"], prefix="rta", dummy_na=False).astype(int)
    df = pd.concat([df, rta_dummies], axis=1)

    return df, trade_vocab, obj_vocab, list(rta_dummies.columns)

merged_main,   tv_m, ov_m, rv_m = add_sanction_features(merged_main)
merged_robust, tv_r, ov_r, rv_r = add_sanction_features(merged_robust)

# --- verify both specs produced the SAME feature vocabulary (critical for comparability) ---
print("descr tokens:", tv_m)
print("objective tokens:", ov_m)
print("rta dummies:", rv_m)
print("main vocab == robust vocab:", (tv_m==tv_r and ov_m==ov_r and rv_m==rv_r))

descr tokens: ['exp_compl', 'exp_part', 'imp_compl', 'imp_part']
objective tokens: ['democracy', 'destab_regime', 'end_war', 'human_rights', 'other', 'policy_change', 'prevent_war', 'territorial_conflict', 'terrorism']
rta dummies: ['rta_Goods', 'rta_Goods & Services', 'rta_Services']
main vocab == robust vocab: True


In [ ]:
# fail loudly if the two specs ever diverge in columns
assert list(merged_main.columns) == list(merged_robust.columns), "main/robust column mismatch!"

In [ ]:
sanction_types = ["arms","military","trade","financial","travel","other"]
sanc_feats = ([f"sanc_{t}" for t in sanction_types]
              + [c for c in merged_main.columns if c.startswith(("descr_","obj_","rta_"))]
              + ["target_mult","sender_mult"])
print(len(sanc_feats), "sanction features:")
print(sanc_feats)

25 sanction features:
['sanc_arms', 'sanc_military', 'sanc_trade', 'sanc_financial', 'sanc_travel', 'sanc_other', 'descr_trade', 'descr_exp_compl', 'descr_exp_part', 'descr_imp_compl', 'descr_imp_part', 'obj_democracy', 'obj_destab_regime', 'obj_end_war', 'obj_human_rights', 'obj_other', 'obj_policy_change', 'obj_prevent_war', 'obj_territorial_conflict', 'obj_terrorism', 'rta_Goods', 'rta_Goods & Services', 'rta_Services', 'target_mult', 'sender_mult']


Descr_trade is still there. No problem - will drop it with all other initial variables.

In [ ]:
# Drop initial variables (already encoded into dummies)
for df_ in (merged_main, merged_robust):
    df_.drop(columns=["descr_trade", "objective", "rta"], inplace=True, errors="ignore")

In [ ]:
# 2021 has no BACI trade at all (empty tail year) -> drop it
merged_main   = merged_main[merged_main["year"] <= 2020].reset_index(drop=True)
merged_robust = merged_robust[merged_robust["year"] <= 2020].reset_index(drop=True)

print("main years now:", merged_main["year"].min(), "-", merged_main["year"].max())
print("main shape:", merged_main.shape)

main years now: 1995 - 2020
main shape: (1528956, 54)


### Zero-trade observations

In [ ]:
# original dataset: trade properties
print("orig total:", len(merged_main))
print("orig trade > 0:", int((merged_main["trade"]>0).sum()))
print("orig trade == 0:", int((merged_main["trade"]==0).sum()))
print("orig trade NaN:", int(merged_main["trade"].isna().sum()))
print("orig sanctioned:", int(merged_main["sanctioned_any"].sum()))

orig total: 1528956
orig trade > 0: 694828
orig trade == 0: 0
orig trade NaN: 834128
orig sanctioned: 94192


***Why zero-trade rows matter***

BACI records only **positive trade flows**. In the CEPII grid, dyad-years with no recorded trade appear as **missing (NaN)** rather than as explicit zeros. These missing values are exactly the **extensive-margin zeros** that sanctions may affect most strongly — for example, **USA → Iran after 2012**, where trade collapses. Left as NaN, a PPML model would ignore them; coded as `0`, they become informative observations.

***What we do***

We convert implicit non-trading dyad-years into explicit zeros — but only after removing rows for country pairs that did not yet (or no longer) exist:

`trade = NaN  →  trade = 0`  (for valid, existing dyad-years)

***Design notes***

1. **Country existence is enforced with the CEPII Countries file.**
   The CEPII grid does **not** gate existence on its own — e.g. **South Sudan appears from 1995** even though it only became a state in 2011. Filling NaN with 0 blindly would fabricate false zeros for these pre-existence years. We therefore use CEPII `first_year` / `last_year` to drop dyad-years where either country did not yet exist (or had ceased to exist), and only then recode the remaining NaN trade as 0.

2. **The pair universe is the full (existence-valid) CEPII grid.**
   We keep all dyad-years CEPII provides for existing country pairs, including pairs that never trade. A permanent zero is itself a meaningful outcome for a sanctions analysis, and dropping such pairs would discard sanctioned dyads that never traded.

3. **Time-invariant controls are already present.**
   Because these rows already exist in the CEPII grid (only `trade` was missing), gravity controls such as distance, contiguity, common language, and colonial ties are already populated — no backfilling is required.

In [ ]:
# ============================================================
# ZERO-TRADE with explicit BEFORE/AFTER comparison.
# Drops pre/post-existence rows (CEPII doesn't gate existence),
# then recodes remaining NaN trade -> 0.
# ============================================================

countries = pd.read_stata(import_sup_path + "Countries_V202211.dta")
lo = dict(zip(countries["iso3"], countries["first_year"].fillna(-9999)))
hi = dict(zip(countries["iso3"], countries["last_year"].fillna(9999)))

def trade_stats(df, label):
    """Return a dict of key trade/sanction counts for comparison."""
    return {
        "label":        label,
        "total":        len(df),
        "trade > 0":    int((df["trade"] > 0).sum()),
        "trade == 0":   int((df["trade"] == 0).sum()),
        "trade NaN":    int(df["trade"].isna().sum()),
        "sanctioned":   int(df["sanctioned_any"].sum()),
    }

def drop_nonexistent_then_zero(df):
    df = df.copy()
    exp_lo = df["exp_iso3"].map(lo).fillna(-9999); exp_hi = df["exp_iso3"].map(hi).fillna(9999)
    imp_lo = df["imp_iso3"].map(lo).fillna(-9999); imp_hi = df["imp_iso3"].map(hi).fillna(9999)
    both_exist = ((df["year"] >= exp_lo) & (df["year"] <= exp_hi) &
                  (df["year"] >= imp_lo) & (df["year"] <= imp_hi))
    df = df[both_exist].reset_index(drop=True)
    df["trade"] = df["trade"].fillna(0)
    return df

# --- snapshot BEFORE ---
before_main = trade_stats(merged_main, "BEFORE")

# --- transform ---
merged_main   = drop_nonexistent_then_zero(merged_main)
merged_robust = drop_nonexistent_then_zero(merged_robust)

# --- snapshot AFTER ---
after_main = trade_stats(merged_main, "AFTER ")

# --- explicit side-by-side comparison ---
comparison = pd.DataFrame([before_main, after_main]).set_index("label")
comparison.loc["DELTA"] = comparison.loc["AFTER "] - comparison.loc["BEFORE"]
print(comparison.to_string())

          total  trade > 0  trade == 0  trade NaN  sanctioned
label                                                        
BEFORE  1528956     694828           0     834128       94192
AFTER   1404170     688811      715359          0       88759
DELTA   -124786      -6017      715359    -834128       -5433


In [ ]:
ssd = merged_main[(merged_main["exp_iso3"]=="SSD")|(merged_main["imp_iso3"]=="SSD")]
print("\nSSD year range now:", ssd["year"].min(), "-", ssd["year"].max())  # 2011 - 2020


SSD year range now: 2011 - 2020


### Export

In [ ]:
merged_main.to_csv(export_path + "merged_main_final.csv", index=False)
merged_robust.to_csv(export_path + "merged_robust_final.csv", index=False)
print("saved final panels")

saved final panels
